# Continue an existing parallel dataset — add recordings, same network

Extend an **already-generated** session with **more recording trials**, reusing the
*exact* fixed network from the original run — **without** re-doing the recordings
already on disk.

**How it works.** Each recording is fully determined by the fixed wiring plus its
per-recording noise reseed `rec_idx` (`Random123(base, gid, rec_idx)`). This notebook
loads the session's saved `_worker_config.pkl` (which holds the original topology, so
the wiring is byte-identical) and spawns the same per-recording worker
(`parallel_dataset.py <cfg> <rec_idx> <summary>`) for the **new indices only**. So
recording *k* is a genuinely new trial on the same graph, and existing files are
never touched.

> Target: the normal-state flagship `NEURON data parallel/normal/20260721_163430`
> (926 cells; recordings 000–049 already present), adding the next 50 (indices
> 050–099). **File indices are 0-based**, so the “51st–100th” recordings are
> `recording050 … recording099`.

In [ ]:
import os, sys, json, glob, time, pickle, subprocess

REPO_ROOT = os.path.abspath('..')
for p in (REPO_ROOT, os.path.join(REPO_ROOT, 'inference')):
    if p not in sys.path:
        sys.path.insert(0, p)
from neuron_simulation import parallel_dataset as pds

# ================== WHAT TO EXTEND ==================
# Existing session to continue (its saved config -> the SAME network is reused).
SESSION_DIR = os.path.abspath(os.path.join('NEURON data parallel', 'normal', '20260721_163430'))

# NEW recordings to add. Indices are 0-based: existing are recording000..049, so the
# "51st..100th" recordings are indices 50..99 (STOP_INDEX is EXCLUSIVE).
START_INDEX = 50
STOP_INDEX  = 100          # exclusive -> generates recording050 .. recording099

# Raster style for the NEW figures (matches the regenerated 000-049 flagship figures).
RASTER_DOT_SIZE     = 4.0
RASTER_BURST_COUNT  = False

# Parallelism / memory. record_voltage=True is inherited from the original run, so each
# worker is large -- keep PER_WORKER_GB ~1.0 (same as the main generation notebook).
MAX_WORKERS   = None
PER_WORKER_GB = 1.0
HEADROOM_GB   = 4.0
POLL_S        = 1.0

assert os.path.isdir(SESSION_DIR), 'session not found: %s' % SESSION_DIR
print('continue session :', SESSION_DIR)
print('new indices      : %d .. %d  (%d recordings)' % (
    START_INDEX, STOP_INDEX - 1, STOP_INDEX - START_INDEX))

## 1. Build the continuation config (reuse the SAME network)

Load the original run's `_worker_config.pkl` (holds the exact topology) and only
re-point it at this folder + set the raster style. The original pkl is left untouched;
a separate `_worker_config_continue.pkl` is written for the workers.

In [ ]:
with open(os.path.join(SESSION_DIR, '_worker_config.pkl'), 'rb') as f:
    cfg = pickle.load(f)

cfg['save_dir']  = os.path.dirname(SESSION_DIR)      # absolute -> CWD-independent worker
cfg['timestamp'] = os.path.basename(SESSION_DIR)
cfg['raster_dot_size']         = RASTER_DOT_SIZE
cfg['raster_show_burst_count'] = RASTER_BURST_COUNT

CONT_PKL = os.path.join(SESSION_DIR, '_worker_config_continue.pkl')
with open(CONT_PKL, 'wb') as f:
    pickle.dump(cfg, f)

topo = cfg['topology']
print('state=%s | sahp_ainc_slow=%.3f | record_voltage=%s | %.0fs per recording' % (
    cfg['state_name'], cfg['build_kwargs'].get('sahp_ainc_slow', float('nan')),
    cfg['record_voltage'], cfg['recording_duration'] / 1000))
print('SAME network: N=%d cells, %d edges  (reused from saved config)' % (
    topo['n_neurons'], len(topo['connections'])))

## 2. Preview the worker count (memory-aware)

In [ ]:
n_new = STOP_INDEX - START_INDEX
workers, free_gb, cpu = pds.pick_worker_count(
    n_new, per_worker_gb=PER_WORKER_GB, headroom_gb=HEADROOM_GB, max_workers=MAX_WORKERS)
print('cpu=%d | free=%.1f GB | -> %d concurrent workers for %d new recordings' % (
    cpu, free_gb, workers, n_new))
print('est added RAM ~%.1f GB peak' % (workers * PER_WORKER_GB))

## 3. Generate the new recordings

Spawns the per-recording worker for the new indices only. **Idempotent**: any index
whose `recordingNNN.npz` already exists is skipped, so re-running never overwrites
existing recordings.

In [ ]:
worker_py = os.path.abspath(pds.__file__)

todo = [i for i in range(START_INDEX, STOP_INDEX)
        if not os.path.exists(os.path.join(SESSION_DIR, 'recording%03d.npz' % i))]
already = sorted(set(range(START_INDEX, STOP_INDEX)) - set(todo))
if already:
    print('skip (already present):', already)
print('generating %d recordings: %s%s' % (len(todo), todo[:8], ' ...' if len(todo) > 8 else ''))

def _launch(idx):
    summ = os.path.join(SESSION_DIR, '_summary_%03d.json' % idx)
    logf = open(os.path.join(SESSION_DIR, 'recording%03d.log' % idx), 'w')
    proc = subprocess.Popen([sys.executable, worker_py, CONT_PKL, str(idx), summ],
                            stdout=logf, stderr=subprocess.STDOUT)
    return proc, summ, logf

pending, running, results = list(todo), {}, {}
t0 = time.time()
while pending or running:
    while pending and len(running) < workers:
        idx = pending.pop(0)
        running[idx] = _launch(idx)
        print('  [launch] recording %03d  (%d running, %d queued)' % (
            idx, len(running), len(pending)), flush=True)
    done = [i for i, (p, _, _) in running.items() if p.poll() is not None]
    for idx in done:
        proc, summ, logf = running.pop(idx)
        logf.close()
        if proc.returncode == 0 and os.path.exists(summ):
            results[idx] = json.load(open(summ))
        else:
            results[idx] = {'index': idx, 'success': False,
                            'error': 'worker exit=%s (see recording%03d.log)' % (proc.returncode, idx)}
        r = results[idx]
        print('  [done]   recording %03d  %s  spikes=%s bursts=%s  (%.0fs)' % (
            idx, 'OK' if r.get('success') else 'FAIL',
            r.get('num_spikes', '?'), r.get('n_bursts', '?'), time.time() - t0), flush=True)
    if running and not done:
        time.sleep(POLL_S)

n_ok = sum(1 for r in results.values() if r.get('success'))
print('DONE: %d/%d new recordings OK in %.1f min' % (n_ok, len(todo), (time.time() - t0) / 60))

## 4. Refresh `session_metadata.json`

Rebuild the recordings list from every per-recording summary so the metadata reflects
ALL recordings now on disk (keeps 000–049, adds the new ones).

In [ ]:
meta_path = os.path.join(SESSION_DIR, 'session_metadata.json')
meta = json.load(open(meta_path, encoding='utf-8')) if os.path.exists(meta_path) else {}

recs = []
for sf in sorted(glob.glob(os.path.join(SESSION_DIR, '_summary_*.json'))):
    try:
        recs.append(json.load(open(sf)))
    except Exception as e:
        print('  [warn] skip %s: %s' % (os.path.basename(sf), e))
recs.sort(key=lambda r: r.get('index', 10**9))

meta['recordings']   = recs
meta['n_recordings'] = len(recs)
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, default=str)

n_ok = sum(1 for r in recs if r.get('success'))
print('session_metadata.json -> %d recordings (%d OK); indices %s..%s' % (
    len(recs), n_ok, recs[0].get('index'), recs[-1].get('index')))

## Notes

- **Same graph, new trials.** The wiring comes from the original run's saved
  `_worker_config.pkl`, so recordings 050–099 sit on the identical network as
  000–049 — only the noise reseed (`rec_idx`) differs, exactly as within the
  original batch.
- **Idempotent.** Any index whose `recordingNNN.npz` already exists is skipped, so
  re-running never clobbers existing recordings.
- **Add even more later.** Bump `START_INDEX` / `STOP_INDEX` (e.g. 100 → 150) and run
  again — the same network is reused every time.
- **Metadata caveat.** The refreshed `session_metadata.json` updates `n_recordings`
  and the `recordings` list; the older `parameters` / `deviations_from_default` blocks
  still quote the original `n_recordings` (documentation only).